## **Experiment 06:** Implementation and Performance Evaluation of Perceptron and Multilayer Perceptron (MLP) for Classification

**Course:** ICE 4111: Artificial Intelligence Lab  
**Program:** Bachelor of Science in Information and Communication Engineering (BSc in ICE)  
**Instructor:** Md. Ryhan Uddin

**Student Name:** _Prity Ghosh_____________________________  
**Student ID:** __0812310105171016____________________________  
**Section/Batch:** ___14th ___________________________  
**Date of Submission:** __04.10.2026____________________________  

> **Runs on Google Colab:** click *Runtime → Run all*. No manual installation or file upload is needed (the dataset downloads automatically).

### **Quick Overview**

Build a single-layer **Perceptron** and a **Multilayer Perceptron (MLP)** in TensorFlow/Keras, train both on the same standardized dataset with a validation set and early stopping, and compare them using learning curves, confusion matrices, and ROC curves. Plotly charts and an ipywidgets slider let you hover for exact values and move a classification-threshold slider to see the MLP's predictions change live.

### **Learning Objectives**

- Build and train a single-layer perceptron for binary classification.
- Build and train an MLP with hidden layers and non-linear activations.
- Use a validation split and early stopping to avoid overfitting.
- Compare learning curves, confusion matrices, and ROC-AUC between the two models.
- Explore how the classification threshold affects precision, recall, and the confusion matrix.

## **Theory**

### Introduction
The perceptron is one of the earliest neural network models. It computes a weighted sum of inputs and applies a sigmoid-like activation to make a binary decision. The MLP adds hidden layers so the network can model non-linear relationships.

### Mathematical Background
For input $x$, weights $w$, and bias $b$: $z = w^T x + b$, and $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$.

Training minimizes binary cross-entropy:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

In an MLP the hidden output is $h = \phi(W_1 x + b_1)$ with $\phi$ = ReLU, and $\hat{y} = \sigma(W_2 h + b_2)$. Gradients flow back through all layers by backpropagation.

### Important Concepts
- A perceptron has no hidden layer, so it can only represent a **linear** decision boundary.
- An MLP can model **non-linear** boundaries.
- **Early stopping** halts training when validation loss stops improving for `patience` epochs and restores the best weights.
- The **classification threshold** (default 0.5) converts probability to class. Lowering it raises recall and lowers precision; raising it does the opposite.

### Advantages and Disadvantages
**Advantages:** perceptron is simple and fast; MLP is far more expressive; early stopping controls overfitting.  
**Disadvantages:** perceptron only solves linearly separable problems; MLP needs more data, tuning, and computation.

```mermaid
flowchart LR
    X[Input Features] --> I[Impute + Standardize]
    I --> P[Perceptron: Single Dense Layer]
    I --> H1[Hidden Layer 1: ReLU]
    H1 --> H2[Hidden Layer 2: ReLU]
    H2 --> O[MLP Output: Sigmoid]
    P --> E[Evaluation and Comparison]
    O --> E
```

## **Required Software and Libraries**
Python 3, NumPy, Pandas, Matplotlib, TensorFlow/Keras, Scikit-learn, Plotly, ipywidgets. All are pre-installed on Google Colab. Run the first code cell; if it prints `Setup complete.`, you are ready.

## **Dataset Description**

This experiment uses the **Pima Indians Diabetes dataset** (768 patients, loaded from OpenML).

- **Features (8):** pregnancies, plasma glucose, diastolic blood pressure, skin-fold thickness, insulin, BMI, diabetes pedigree function, age.
- **Target:** `0` = non-diabetic, `1` = diabetic.
- **Data quality note:** in `glucose`, `blood_pressure`, `skin_thickness`, `insulin`, and `bmi`, a value of `0` is physiologically impossible and means *missing*. We convert these to `NaN` and impute with the **training-set median** (fitted on training data only, to avoid data leakage).
- **Why this dataset:** it is noisier and less cleanly separable than a textbook benchmark, so the differences between models and thresholds are more realistic.

## **Experimental Procedure**
1. Load the dataset, fix invalid zeros, split into train/validation/test, impute, and standardize.
2. Build and train a single-neuron perceptron with early stopping.
3. Build and train an MLP with two hidden layers with early stopping.
4. Evaluate both on the test set (accuracy, precision, recall, F1, ROC-AUC).
5. Compare learning curves, confusion matrices, and ROC curves.
6. Use the threshold slider to explore the precision/recall trade-off for the MLP.

### Student Checkpoint
- Why use a separate validation set instead of only train/test?
- At what epoch did early stopping halt each model?
- What happens to the MLP's precision and recall as you move the threshold away from 0.5?

## **Source Code**
Run the cells one by one and read the output after each.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider

from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Setup complete. You can start the experiment.")

Setup complete. You can start the experiment.


In [2]:
# Load the Pima Indians Diabetes dataset from OpenML
raw = fetch_openml(name="diabetes", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()

# Standard, readable column names
df.columns = ["pregnancies", "glucose", "blood_pressure", "skin_thickness",
              "insulin", "bmi", "pedigree", "age", "outcome"]
df["outcome"] = (df["outcome"] == "tested_positive").astype(int)   # 1 = diabetic

# Zeros are impossible values in these columns -> treat as missing
zero_as_missing = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

print("Missing values per column:\n", df.isna().sum(), "\n")
df.head()

Missing values per column:
 pregnancies         0
glucose             5
blood_pressure     35
skin_thickness    227
insulin           374
bmi                11
pedigree            0
age                 0
outcome             0
dtype: int64 



,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,pedigree,age,outcome
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50,1
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31,0
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


In [3]:
X = df.drop(columns="outcome")
y = df["outcome"]
labels = ["non-diabetic", "diabetic"]

print("Shape:", X.shape)
print("Class counts:\n", y.value_counts().sort_index())
print("Class names:", labels)

Shape: (768, 8)
Class counts:
 outcome
0    500
1    268
Name: count, dtype: int64
Class names: ['non-diabetic', 'diabetic']


In [4]:
# Train / Validation / Test split, then impute (median) and standardize
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

imputer = SimpleImputer(strategy="median").fit(X_train)     # fit on training data only
scaler = StandardScaler().fit(imputer.transform(X_train))

def prep(a):
    return scaler.transform(imputer.transform(a))

X_train_scaled, X_val_scaled, X_test_scaled = prep(X_train), prep(X_val), prep(X_test)

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (491, 8)
Validation shape: (123, 8)
Test shape: (154, 8)


In [5]:
# Build the Perceptron
perceptron_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
perceptron_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                         loss="binary_crossentropy", metrics=["accuracy"])
perceptron_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the Perceptron with Early Stopping
perceptron_history = perceptron_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100, batch_size=32, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
)
print(f"Perceptron training finished after {len(perceptron_history.history['loss'])} epochs.")

Perceptron training finished after 22 epochs.


In [7]:
# Evaluate the Perceptron on the Test Set
perceptron_test_probs = perceptron_model.predict(X_test_scaled, verbose=0).ravel()
perceptron_test_pred = (perceptron_test_probs >= 0.5).astype(int)

print(f"Perceptron Test Accuracy : {accuracy_score(y_test, perceptron_test_pred):.4f}")
print(classification_report(y_test, perceptron_test_pred, target_names=labels))

Perceptron Test Accuracy : 0.6948
              precision    recall  f1-score   support

non-diabetic       0.75      0.80      0.77       100
    diabetic       0.57      0.50      0.53        54

    accuracy                           0.69       154
   macro avg       0.66      0.65      0.65       154
weighted avg       0.69      0.69      0.69       154



In [8]:
# Build the MLP
mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
mlp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss="binary_crossentropy", metrics=["accuracy"])
mlp_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Train the MLP with Early Stopping
mlp_history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200, batch_size=32, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
)
print(f"MLP training finished after {len(mlp_history.history['loss'])} epochs.")

MLP training finished after 42 epochs.


In [10]:
# Evaluate the MLP on the Test Set
mlp_test_probs = mlp_model.predict(X_test_scaled, verbose=0).ravel()
mlp_test_pred = (mlp_test_probs >= 0.5).astype(int)

print(f"MLP Test Accuracy : {accuracy_score(y_test, mlp_test_pred):.4f}")
print(classification_report(y_test, mlp_test_pred, target_names=labels))

MLP Test Accuracy : 0.7208
              precision    recall  f1-score   support

non-diabetic       0.75      0.85      0.80       100
    diabetic       0.63      0.48      0.55        54

    accuracy                           0.72       154
   macro avg       0.69      0.67      0.67       154
weighted avg       0.71      0.72      0.71       154



In [11]:
# Compare Both Models on Test Metrics
comparison = []
for name, yp, ypr in [("Perceptron", perceptron_test_pred, perceptron_test_probs),
                      ("MLP", mlp_test_pred, mlp_test_probs)]:
    comparison.append([name,
                       accuracy_score(y_test, yp),
                       precision_score(y_test, yp, zero_division=0),
                       recall_score(y_test, yp, zero_division=0),
                       f1_score(y_test, yp, zero_division=0),
                       roc_auc_score(y_test, ypr)])

comparison_df = pd.DataFrame(comparison,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]).round(4)
comparison_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Perceptron,0.6948,0.5745,0.5000,0.5347,0.8131
1,MLP,0.7208,0.6341,0.4815,0.5474,0.8104


In [12]:
# Interactive Learning Curves: Accuracy (hover for exact values)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Accuracy", "MLP Accuracy"))
fig.add_trace(go.Scatter(y=perceptron_history.history["accuracy"], mode="lines", name="Train",
                         line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_accuracy"], mode="lines", name="Validation",
                         line=dict(color="#e76f51")), row=1, col=1)
fig.add_trace(go.Scatter(y=mlp_history.history["accuracy"], mode="lines", name="Train",
                         line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_accuracy"], mode="lines", name="Validation",
                         line=dict(color="#e76f51"), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Accuracy")
fig.update_layout(title="Training vs Validation Accuracy", height=450, width=950, template="plotly_white")
fig.show()

In [13]:
# Interactive Learning Curves: Loss (hover for exact values)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Loss", "MLP Loss"))
fig.add_trace(go.Scatter(y=perceptron_history.history["loss"], mode="lines", name="Train",
                         line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_loss"], mode="lines", name="Validation",
                         line=dict(color="#e76f51")), row=1, col=1)
fig.add_trace(go.Scatter(y=mlp_history.history["loss"], mode="lines", name="Train",
                         line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_loss"], mode="lines", name="Validation",
                         line=dict(color="#e76f51"), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Binary Cross-Entropy Loss")
fig.update_layout(title="Training vs Validation Loss", height=450, width=950, template="plotly_white")
fig.show()

In [14]:
# Interactive Confusion Matrices
perceptron_cm = confusion_matrix(y_test, perceptron_test_pred)
mlp_cm = confusion_matrix(y_test, mlp_test_pred)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron", "MLP"))
fig.add_trace(go.Heatmap(z=perceptron_cm, x=labels, y=labels, colorscale="Blues",
                         text=perceptron_cm, texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=mlp_cm, x=labels, y=labels, colorscale="Blues",
                         text=mlp_cm, texttemplate="%{text}", showscale=False), row=1, col=2)
fig.update_xaxes(title_text="Predicted Label")
fig.update_yaxes(title_text="True Label", autorange="reversed")
fig.update_layout(title="Confusion Matrix Comparison", height=450, width=950, template="plotly_white")
fig.show()

In [15]:
# Interactive ROC Curve Comparison
fpr_p, tpr_p, _ = roc_curve(y_test, perceptron_test_probs)
fpr_m, tpr_m, _ = roc_curve(y_test, mlp_test_probs)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_p, y=tpr_p, mode="lines",
    name=f"Perceptron (AUC = {roc_auc_score(y_test, perceptron_test_probs):.3f})",
    line=dict(color="#2a9d8f"), hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=fpr_m, y=tpr_m, mode="lines",
    name=f"MLP (AUC = {roc_auc_score(y_test, mlp_test_probs):.3f})",
    line=dict(color="#e76f51"), hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random Guess",
    line=dict(color="gray", dash="dash")))
fig.update_layout(title="ROC Curve Comparison", xaxis_title="False Positive Rate",
                  yaxis_title="True Positive Rate", height=500, width=650, template="plotly_white")
fig.show()

In [16]:
# Interactive Threshold Explorer for the MLP
# Move the slider to change the threshold and watch the metrics and confusion matrix update.

def explore_threshold(threshold=0.50):
    y_pred_t = (mlp_test_probs >= threshold).astype(int)
    print(f"Threshold : {threshold:.2f}")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred_t):.4f}")
    print(f"Precision : {precision_score(y_test, y_pred_t, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_test, y_pred_t, zero_division=0):.4f}")
    print(f"F1-score  : {f1_score(y_test, y_pred_t, zero_division=0):.4f}")

    cm = confusion_matrix(y_test, y_pred_t, labels=[0, 1])
    fig = go.Figure(data=go.Heatmap(z=cm, x=labels, y=labels, colorscale="Blues",
                                    text=cm, texttemplate="%{text}", showscale=False))
    fig.update_xaxes(title_text="Predicted Label")
    fig.update_yaxes(title_text="True Label", autorange="reversed")
    fig.update_layout(title=f"MLP Confusion Matrix at Threshold = {threshold:.2f}",
                      height=420, width=420, template="plotly_white")
    fig.show()

interact(explore_threshold,
         threshold=FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                               description="Threshold", continuous_update=False));

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='Threshold', max=0.95, min=0…

## **Expected Output**
- Dataset shape, class counts, and missing-value counts.
- Train/validation/test shapes after imputation and standardization.
- Model summaries and the epoch at which early stopping triggered.
- Test accuracy and classification report for both models, plus a comparison table.
- Interactive accuracy/loss curves, confusion matrices, and ROC curves.
- A working threshold slider.

Include the important outputs and a short note on what you observed with the slider in your report.

## **Performance Evaluation**
Use accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices. Learning curves show overfitting (validation loss rising while training loss falls) or underfitting (both staying high). Use the threshold explorer to judge the trade-off beyond the default 0.5 cutoff.

## **Discussion**
*(Write this in your own words after running the notebook, using your actual numbers.)* Guiding points:

- Which model scored higher on accuracy, F1, and ROC-AUC? Is the difference large, or within what you would expect from a small test set (154 patients)?
- This dataset is noisy with overlapping classes, so a more complex MLP does not automatically beat a linear perceptron. Do your learning curves show the MLP overfitting?
- Where did early stopping halt each model, and what does it say about convergence?
- In diabetes screening a **false negative** (missed diabetic patient) is usually costlier than a false alarm. Which threshold would you choose, and why?

## **Viva Questions**

1. **Q:** What is the perceptron model?  
   **A:** A single-layer neural classifier that computes a weighted sum of inputs and outputs a binary decision.
2. **Q:** What makes an MLP more powerful than a perceptron?  
   **A:** Hidden layers with non-linear activations model non-linearly-separable relationships.
3. **Q:** What is backpropagation?  
   **A:** A method for computing the gradient of the loss with respect to every weight, layer by layer.
4. **Q:** Why standardize features?  
   **A:** To put features on a comparable scale, improving training stability and speed.
5. **Q:** What is early stopping, and what does `restore_best_weights` do?  
   **A:** It halts training when validation performance stops improving for `patience` epochs and reloads the best epoch's weights.
6. **Q:** Why is sigmoid used in the output layer?  
   **A:** It maps the output to a probability between 0 and 1.
7. **Q:** Why a validation set separate from the test set?  
   **A:** To monitor generalization and tune training without leaking test information.
8. **Q:** Why can a perceptron fail on some datasets?  
   **A:** It can only learn a linear decision boundary.
9. **Q:** What happens to precision and recall as the threshold is lowered?  
   **A:** Recall tends to rise and precision tends to fall.
10. **Q:** Why did we replace zeros with the training median in this dataset?  
    **A:** Zeros in glucose, blood pressure, skin thickness, insulin, and BMI are impossible values meaning "missing"; using the training-set median avoids leaking test information.

## **Exercises**
1. Increase the hidden layers or units in the MLP and compare.
2. Try a different learning rate and observe convergence.
3. Replace `relu` with `tanh` and evaluate.
4. Use the slider to find the threshold that maximizes F1 and report it. Then find one that gives recall above 0.85 and note the cost in precision.
5. Add a `Dropout` layer and check the train/validation gap.
6. Repeat the workflow on the Iris dataset (softmax output) for a multiclass experiment.

## **Lab Report Submission Checklist**
- Completed cover information.
- Objective and theory summary in your own words.
- Important outputs, including the metric comparison table.
- Screenshots/descriptions of the curves, confusion matrices, ROC, and threshold explorer.
- Discussion of which model performed best and why.
- Viva answers and completed exercises.

**Reminder:** A good report explains what the outputs mean, not just screenshots.